In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
from torch_geometric.loader import DataLoader
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.utils import dropout_edge
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# --- 1. Load Data & Split (Stratified) ---
saved_graphs = torch.load("wsi_spatial_graphs.pt", weights_only=False)
dataset = [Data(x=g["x"], edge_index=g["edge_index"], y=g["y"]) for g in saved_graphs.values()]

all_labels = [data.y.item() for data in dataset]
train_val_dataset, test_dataset = train_test_split(dataset, test_size=0.2, stratify=all_labels, random_state=42)
train_val_labels = [data.y.item() for data in train_val_dataset]
train_dataset, val_dataset = train_test_split(train_val_dataset, test_size=0.2, stratify=train_val_labels, random_state=42)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# --- 2. Architecture ---
class VisualOnlyGNN(torch.nn.Module):
    def __init__(self, in_channels=1024, hidden_channels=256, num_classes=4, 
                 dropout_rate=0.1130, dropedge_rate=0.2291): # <-- OPTIMIZED PARAMETERS
        super(VisualOnlyGNN, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels // 2)
        self.dropout_rate = dropout_rate
        self.dropedge_rate = dropedge_rate
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(hidden_channels // 2, 64),
            torch.nn.ReLU(),
            torch.nn.Dropout(self.dropout_rate), 
            torch.nn.Linear(64, num_classes)
        )
    
    def forward(self, x, edge_index, batch):
        edge_index, _ = dropout_edge(edge_index, p=self.dropedge_rate, training=self.training)
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout_rate, training=self.training)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = global_mean_pool(x, batch)
        out = self.classifier(x)
        return out

# --- 3. Setup ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_labels = [data.y.item() for data in train_dataset]
weights_tensor = torch.tensor(compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels), dtype=torch.float).to(device)
criterion = torch.nn.CrossEntropyLoss(weight=weights_tensor)

model = VisualOnlyGNN().to(device)

# <-- OPTIMIZED LEARNING RATE & WEIGHT DECAY
optimizer = torch.optim.Adam(model.parameters(), lr=0.000497, weight_decay=0.000252) 

# --- 4. Final Training Loop with Early Stopping ---
epochs = 30
best_val_loss = float('inf')
patience, patience_counter = 5, 0

print("Training Final Optimized Baseline Model...")
for epoch in range(epochs):
    model.train()
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        loss = criterion(model(batch.x, batch.edge_index, batch.batch), batch.y)
        loss.backward()
        optimizer.step()
        
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for val_batch in val_loader:
            val_batch = val_batch.to(device)
            val_loss += criterion(model(val_batch.x, val_batch.edge_index, val_batch.batch), val_batch.y).item()
    
    avg_val_loss = val_loss / len(val_loader)
    print(f"Epoch {epoch+1:02d} | Val Loss: {avg_val_loss:.4f}")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "best_visual_gnn.pth")
        patience_counter = 0 
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping triggered.")
            break

print("Training complete! Model saved as 'best_visual_gnn.pth'")

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Assuming 'model', 'test_loader', and 'device' are still in memory from your training script

def evaluate_best_model(model, test_loader, device, model_weights_path):
    print(f"Loading best weights from: {model_weights_path}...")
    model.load_state_dict(torch.load(model_weights_path, weights_only=True))
    model.to(device)
    model.eval()

    all_predictions = []
    all_true_labels = []

    with torch.no_grad(): # Use no_grad for broader compatibility
        for batch in test_loader:
            batch = batch.to(device)
            
            # Forward pass
            outputs = model(batch.x, batch.edge_index, batch.batch)
            _, predicted = torch.max(outputs.data, 1)

            all_predictions.extend(predicted.cpu().numpy())
            all_true_labels.extend(batch.y.cpu().numpy())

    # Calculate overall accuracy
    acc = accuracy_score(all_true_labels, all_predictions)
    print(f"\n======================================")
    print(f"OVERALL TEST ACCURACY: {acc * 100:.2f}%")
    print(f"======================================\n")
    
    # Generate Classification Report
    target_names = ["Non-Tumor", "Viable", "Non-Viable-Tumor", "viable: non-viable"]
    print("Classification Report:")
    print(classification_report(all_true_labels, all_predictions, 
                                labels=[0, 1, 2, 3], 
                                target_names=target_names, 
                                zero_division=0))

    # Generate Confusion Matrix
    cm = confusion_matrix(all_true_labels, all_predictions, labels=[0, 1, 2, 3])
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=target_names, yticklabels=target_names)
    plt.xlabel('Predicted Classification')
    plt.ylabel('Actual Classification')
    plt.title('Best Visual GNN Evaluation Matrix')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig('best_visual_gnn_matrix.png')
    plt.show()

# Run the evaluation!
evaluate_best_model(model, test_loader, device, "best_visual_gnn.pth")

Hyper Parameter Optimization Script using Optuna:

In [ ]:
import torch
import torch.nn.functional as F
import random
import numpy as np
import optuna
from torch_geometric.loader import DataLoader
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.utils import dropout_edge
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# ==========================================
# 1. Load Data
# ==========================================
print("Loading graphs for HPO...")
saved_graphs = torch.load("wsi_spatial_graphs.pt", weights_only=False)

dataset = []
for image_id, graph_dict in saved_graphs.items():
    data = Data(
        x=graph_dict["x"], 
        edge_index=graph_dict["edge_index"], 
        y=graph_dict["y"]
    )
    dataset.append(data)

# ==========================================
# 2. Split Data (Train / Val / Test)
# ==========================================
all_labels = [data.y.item() for data in dataset]
train_val_dataset, test_dataset = train_test_split(
    dataset, test_size=0.2, stratify=all_labels, random_state=42
)

train_val_labels = [data.y.item() for data in train_val_dataset]
train_dataset, val_dataset = train_test_split(
    train_val_dataset, test_size=0.2, stratify=train_val_labels, random_state=42
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# ==========================================
# 3. Define the Dynamic GNN Architecture
# ==========================================
class VisualOnlyGNN(torch.nn.Module):
    def __init__(self, in_channels=1024, hidden_channels=256, num_classes=4, 
                 dropout_rate=0.4, dropedge_rate=0.2):
        super(VisualOnlyGNN, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels // 2)
        
        self.dropout_rate = dropout_rate
        self.dropedge_rate = dropedge_rate
        
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(hidden_channels // 2, 64),
            torch.nn.ReLU(),
            torch.nn.Dropout(self.dropout_rate), 
            torch.nn.Linear(64, num_classes)
        )
    
    def forward(self, x, edge_index, batch):
        edge_index, _ = dropout_edge(edge_index, p=self.dropedge_rate, training=self.training)
        
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout_rate, training=self.training)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = global_mean_pool(x, batch)
        out = self.classifier(x)
        return out

# ==========================================
# 4. Setup Globals for Optimization
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

train_labels = [data.y.item() for data in train_dataset]
class_weights = compute_class_weight(
    class_weight='balanced', 
    classes=np.unique(train_labels), 
    y=train_labels
)
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
criterion = torch.nn.CrossEntropyLoss(weight=weights_tensor)

# ==========================================
# 5. The Optuna Objective Function
# ==========================================
def objective(trial):
    print(f"\n" + "="*50)
    print(f"🚀 STARTING TRIAL {trial.number}")
    print(f"="*50)

    # 1. AI decides the parameters for this run
    lr = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)
    dropout_rate = trial.suggest_float("dropout_rate", 0.05, 0.6)
    dropedge_rate = trial.suggest_float("dropedge_rate", 0.05, 0.4)

    # Print Current Trial Stats
    print("🧪 CURRENT TRIAL PARAMETERS:")
    print(f"   Learning Rate: {lr:.6f}")
    print(f"   Weight Decay:  {weight_decay:.6f}")
    print(f"   Dropout:       {dropout_rate:.4f}")
    print(f"   DropEdge:      {dropedge_rate:.4f}")
    
    # 2. Build the model with these parameters
    model = VisualOnlyGNN(dropout_rate=dropout_rate, dropedge_rate=dropedge_rate).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    # 3. Train quickly (15 Epochs per trial is enough for the AI to see the trend)
    epochs = 15
    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch.x, batch.edge_index, batch.batch)
            loss = criterion(out, batch.y)
            loss.backward()
            optimizer.step()
            
    # 4. Evaluate Validation Accuracy
    model.eval()
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for val_batch in val_loader:
            val_batch = val_batch.to(device)
            val_out = model(val_batch.x, val_batch.edge_index, val_batch.batch)
            pred = val_out.argmax(dim=1)
            correct_val += int((pred == val_batch.y).sum())
            total_val += len(val_batch.y)
            
    val_accuracy = correct_val / total_val
    
    print(f"\n📊 TRIAL {trial.number} RESULTS:")
    print(f"   Validation Accuracy: {val_accuracy:.4f}")

    # 5. Fetch and print historical stats (Compare against the best so far)
    try:
        best_trial = trial.study.best_trial
        print(f"\n🏆 PREVIOUS BEST (From Trial {best_trial.number}):")
        print(f"   Best Val Accuracy: {best_trial.value:.4f}")
        print(f"   Best LR:           {best_trial.params['lr']:.6f}")
        print(f"   Best W-Decay:      {best_trial.params['weight_decay']:.6f}")
        print(f"   Best Dropout:      {best_trial.params['dropout_rate']:.4f}")
        print(f"   Best DropEdge:     {best_trial.params['dropedge_rate']:.4f}")
    except ValueError:
        print("\n🏆 PREVIOUS BEST: (This is the first trial!)")

    return val_accuracy

# ==========================================
# 6. Execute the AI Optimizer
# ==========================================
# Suppress Optuna's default noisy logs so our custom dashboard is easy to read
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("🧠 Initializing Bayesian Optimizer...")
study = optuna.create_study(direction="maximize")

# Run 30 different simulations to find the mathematical optimum
study.optimize(objective, n_trials=30)

print("\n" + "*"*50)
print("🎯 OPTIMIZATION COMPLETE!")
print("*"*50)
print(f"Ultimate Validation Accuracy: {study.best_value:.4f}")
print("Ultimate Parameters to plug into your final model:")
for key, value in study.best_params.items():
    if key in ['lr', 'weight_decay']:
        print(f"  {key}: {value:.6f}")
    else:
        print(f"  {key}: {value:.4f}")